In [18]:
import json
import os
from openai import OpenAI
import openai
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import ttest_ind

In [5]:
def get_embedding(text, client, model="text-embedding-3-small"):
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [ ]:
# os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()
baseline_idea_files = [f'ai_scientist_baseline_ideas{i}.json' for i in ['', 2, 3]]
persona_idea_files = [f'ai_scientist_persona_baseline_ideas{i}.json' for i in ['', 2, 3]]
baseline_idea_sets = []
persona_idea_sets = []
for baseline_idea_file, persona_idea_file in zip(baseline_idea_files, persona_idea_files):
    with open(os.path.join('templates/ppo_with_folds', baseline_idea_file), 'r') as f:
        ideas = json.load(f)
        baseline_idea_sets.append(ideas)
    with open(os.path.join('templates/ppo_with_folds', persona_idea_file), 'r') as f:
        ideas = json.load(f)
        persona_idea_sets.append(ideas)

In [14]:
seed_ideas = ['fold_specific_learning_rates', 'input_dependent_hyperplane_combination']
baseline_embeddings = np.array([[get_embedding(idea["Experiment"], client) for idea in ideas if idea["Name"] not in seed_ideas] for ideas in baseline_idea_sets])
persona_embeddings = np.array([[get_embedding(idea["Experiment"], client) for idea in ideas if idea["Name"] not in seed_ideas] for ideas in persona_idea_sets])

In [17]:
baseline_similarities_between_sets = []
persona_similarities_between_sets = []
for i in range(len(baseline_idea_sets)):
    for j in range(i+1, len(baseline_idea_sets)):
        baseline_similarities_between_sets.extend(cosine_similarity(baseline_embeddings[i], baseline_embeddings[j]).flatten())
        persona_similarities_between_sets.extend(cosine_similarity(persona_embeddings[i], persona_embeddings[j]).flatten())
baseline_similarities_between_sets = np.array(baseline_similarities_between_sets)
persona_similarities_between_sets = np.array(persona_similarities_between_sets)
baseline_similarities_within_sets = []
persona_similarities_within_sets = []
for i in range(len(baseline_idea_sets)):
    baseline_similarities = cosine_similarity(baseline_embeddings[i], baseline_embeddings[i])
    upper_triangle_indices = np.triu_indices(baseline_similarities.shape[0], k=1)
    upper_triangle_values = baseline_similarities[upper_triangle_indices]
    baseline_similarities_within_sets.extend(upper_triangle_values)
    persona_similarities = cosine_similarity(persona_embeddings[i], persona_embeddings[i])
    upper_triangle_indices = np.triu_indices(persona_similarities.shape[0], k=1)
    upper_triangle_values = persona_similarities[upper_triangle_indices]
    persona_similarities_within_sets.extend(upper_triangle_values)
baseline_similarities_within_sets = np.array(baseline_similarities_within_sets)
persona_similarities_within_sets = np.array(persona_similarities_within_sets)

In [21]:
# print the means
print("Baseline similarities between sets mean:", np.mean(baseline_similarities_between_sets))
print("Baseline similarities within sets mean:", np.mean(baseline_similarities_within_sets))
print("Persona similarities between sets mean:", np.mean(persona_similarities_between_sets))
print("Persona similarities within sets mean:", np.mean(persona_similarities_within_sets))

Baseline similarities between sets mean: 0.6358354988966156
Baseline similarities within sets mean: 0.6679188226170747
Persona similarities between sets mean: 0.6197248709797344
Persona similarities within sets mean: 0.6359969457992994


In [22]:
baseline = ttest_ind(baseline_similarities_within_sets, baseline_similarities_between_sets, equal_var=False).pvalue
persona = ttest_ind(persona_similarities_within_sets, persona_similarities_between_sets, equal_var=False).pvalue
print("Baseline t-test p-value:", baseline)
print("Persona t-test p-value:", persona)
between_sets = ttest_ind(baseline_similarities_between_sets, persona_similarities_between_sets, equal_var=True).pvalue
within_sets = ttest_ind(baseline_similarities_within_sets, persona_similarities_within_sets, equal_var=True).pvalue
print("Between sets t-test p-value:", between_sets)
print("Within sets t-test p-value:", within_sets)

Baseline t-test p-value: 0.014288234941501447
Persona t-test p-value: 0.24896742498017874
Between sets t-test p-value: 0.09514894525671029
Within sets t-test p-value: 0.0545221856916545
